In [11]:
# statistical dependence of the events  Xt = w1 and Xt+1 = w2 
# C(w)is the absolute frequency and N is the size of the corpus.

In [12]:
from urllib.request import urlopen
from collections import Counter
import math
import string
import re

In [13]:
# Load the corpus
url = "https://raw.githubusercontent.com/coli-saar/cl/main/a1/junglebook.txt"
text = urlopen(url).read().decode("utf-8")

In [14]:
# Preprocess the corpus (Remove metadata, convert to lowercase, remove illustration tags and punctuation)
start = text.find("Now Rann, the Kite")
end = text.find("Transcriber's Notes:")

text = text[start:end]
text = text.lower()

text = re.sub(r"\[illustration\]", "", text)
text = re.sub(r"\[illustration:.*?\]", "", text, flags=re.DOTALL)

for p in string.punctuation:
    text = text.replace(p, " ")

The corpus was converted to lowercase to avoid counting uppercase and lowercase forms as different words. Metadata, illustration markers, and punctuation were removed during preprocessing. Tokenization was then performed by splitting the text at whitespace.

In [15]:
# Tokenization and word count
words = text.split()
print("Corpus size:", len(words))

word_counts = Counter(words)

Corpus size: 52391


In [16]:
# Count successive word pairs (bigrams)
bigrams = []

for i in range(len(words) - 1):
    bigrams.append((words[i], words[i + 1]))

bigram_counts = Counter(bigrams)
print("Number of distinct bigrams:", len(bigram_counts))

Number of distinct bigrams: 28751


In [17]:
# Ignore words occurring fewer than 10 times
valid_words = set()

for word, count in word_counts.items():
    if count >= 10:
        valid_words.add(word)

print("Words occurring at least 10 times:", len(valid_words))

Words occurring at least 10 times: 669


In [18]:
# Calculate PMI values
N = len(words)

pmi_scores = {}

for (w1, w2), pair_count in bigram_counts.items():

    if w1 not in valid_words:
        continue

    if w2 not in valid_words:
        continue

    pmi = math.log(
        (pair_count * N) /
        (word_counts[w1] * word_counts[w2])
    )

    pmi_scores[(w1, w2)] = pmi

PMI was calculated for all successive word pairs in the corpus, and words occurring fewer than 10 times were excluded from the analysis, 
as specifically requested by the task.

In [ ]:
# Highest PMI values
highest_pmi = sorted(
    pmi_scores.items(),
    key=lambda x: x[1],  # sort according to the PMI value
    reverse=True         # sort in descending order
)[:20]

print("TOP 20 PMI WORD PAIRS\n")

for pair, score in highest_pmi:
    print(pair, round(score, 3))

TOP 20 PMI WORD PAIRS

('machua', 'appa') 8.469
('cold', 'lairs') 7.737
('bandar', 'log') 7.433
('petersen', 'sahib') 7.311
('hind', 'legs') 7.16
('fore', 'paws') 7.082
('hind', 'flippers') 7.06
('troop', 'horse') 6.993
('tree', 'tops') 6.954
('master', 'words') 6.928
('bath', 'room') 6.928
('twenty', 'yoke') 6.91
('whole', 'line') 6.89
('years', 'ago') 6.856
('bat', 'bat') 6.764
('waingunga', 'river') 6.692
('killing', 'grounds') 6.677
('council', 'rock') 6.644
('monkey', 'folk') 6.571
('black', 'panther') 6.461


The 20 word pairs with the highest PMI values were extracted and inspected:
the word pairs with the highest PMI values occur together much more frequently than would be expected if the two words 
were statistically independent. It is not surprising that these pairs often correspond to names, fixed expressions, 
and recurring collocations appearing throughout the corpus.

In [20]:
# Lowest PMI values
lowest_pmi = sorted(
    pmi_scores.items(),
    key=lambda x: x[1]
)[:20]

print("LOWEST 20 PMI WORD PAIRS\n")

for pair, score in lowest_pmi:
    print(pair, round(score, 3))

LOWEST 20 PMI WORD PAIRS

('the', 'he') -4.26
('he', 'the') -4.26
('of', 'and') -3.944
('i', 'and') -3.33
('they', 'the') -3.213
('of', 'he') -3.204
('to', 'he') -3.195
('was', 'and') -3.103
('for', 'and') -2.937
('had', 'the') -2.937
('and', 'is') -2.919
('is', 'and') -2.919
('with', 'and') -2.78
('of', 'in') -2.741
('little', 'the') -2.715
('to', 'i') -2.693
('but', 'and') -2.661
('that', 'and') -2.659
('he', 'in') -2.629
('and', 'of') -2.557


The 20 word pairs with the lowest PMI values were also extracted for comparison:
the lowest PMI values, on the other hand, correspond to word pairs that occur less frequently than expected 
from the individual frequencies of the two words.

The results provide evidence against the independence assumption of unigram language models. In this corpus, the probability of a word 
is clearly influenced by the preceding word, meaning that adjacent words in natural language are statistically dependent.
This helps explain why higher-order language models generally outperform unigram models: they can better capture the contextual information that unigram models ignore.